# 01 — Sandbox 10 stocks, FinRL EIIE / PolicyGradient

Train on the sandbox universe, evaluate on the test window, plot cumulative value vs equal-weight and risk parity.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sp500rl.config import load_config
from sp500rl.seed import set_seed

CFG = load_config(ROOT / "configs" / "default.yaml")
SEED = set_seed(int(CFG["seed"]))
print(f"seed={SEED}")
print("train", CFG["dates"]["train_start"], "→", CFG["dates"]["train_end"])
print("test ", CFG["dates"]["test_start"], "→", CFG["dates"]["test_end"])


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sp500rl.data.pipeline import build_panel_from_file
from sp500rl.env.make_env import make_poe
from sp500rl.finrl_bootstrap import import_pg_stack
from sp500rl.baselines.simple import rollout, rollout_equal_weight, rollout_risk_parity
from sp500rl.eval.metrics import compare_rollouts

processed = ROOT / CFG["paths"]["processed_dir"] / "panel_sandbox.parquet"
if processed.exists():
    panel = pd.read_parquet(processed)
    panel["date"] = pd.to_datetime(panel["date"])
else:
    panel = build_panel_from_file(None, cfg=CFG, universe="sandbox", use_synthetic=True)
print("panel", panel.shape, panel["date"].min().date(), "→", panel["date"].max().date())
print("tickers", sorted(panel["tic"].unique()))


PolicyGradient treats `obs` as a tensor and uses the old gym 4-tuple, so `return_last_action=False` and `new_gym_api=False`. Online learning during test is disabled (`learning_rate=0`) for a frozen-policy comparison.

In [ ]:
DRLAgent, EIIE, GPM, PolicyGradient = import_pg_stack()
print("imported", DRLAgent, EIIE, GPM, PolicyGradient)

features = list(CFG["poe"]["features"])
time_window = int(CFG["poe"]["time_window"])
train_env = make_poe(
    panel, CFG, mode="train",
    return_last_action=False, new_gym_api=False,
    cwd=ROOT / "results" / "eiie_train",
)
print("train obs shape", train_env.reset().shape, "action", train_env.action_space.shape)

agent = DRLAgent(train_env)
model = agent.get_model(
    "pg",
    device="cpu",
    model_kwargs={
        "policy": EIIE,
        "lr": 1e-3,
        "batch_size": 64,
    },
    policy_kwargs={
        "initial_features": len(features),
        "k_size": 3,
        "time_window": time_window,
        "device": "cpu",
    },
)
EPISODES = 2  # bump for research runs
print(f"training EIIE for {EPISODES} episodes, seed={SEED}")
DRLAgent.train_model(model, episodes=EPISODES)


In [ ]:
def rollout_trained_pg(model, env):
    # Frozen policy: learning_rate=0 disables online gradient steps in test().
    model.test(env, learning_rate=0.0, online_training_period=10**9)
    return {
        "dates": list(env._date_memory),
        "values": np.asarray(env._asset_memory["final"], dtype=float),
        "returns": np.asarray(env._portfolio_return_memory, dtype=float),
        "actions": np.asarray(env._actions_memory, dtype=float),
        "trf_mu": np.asarray(
            [1.0] * (len(env._asset_memory["final"]) - 1), dtype=float
        ),
        "rewards": np.asarray(env._portfolio_reward_memory, dtype=float),
    }

test_kwargs = dict(
    panel=panel, cfg=CFG, mode="test",
    return_last_action=False, new_gym_api=False,
)
agent_env = make_poe(cwd=ROOT / "results" / "eiie_agent", **test_kwargs)
eq_env = make_poe(cwd=ROOT / "results" / "eiie_eq", **test_kwargs)
rp_env = make_poe(cwd=ROOT / "results" / "eiie_rp", **test_kwargs)

agent_roll = rollout_trained_pg(model, agent_env)
eq_roll = rollout_equal_weight(eq_env)
rp_roll = rollout_risk_parity(rp_env, panel)

summary = compare_rollouts(
    {"EIIE": agent_roll, "equal_weight": eq_roll, "risk_parity": rp_roll}
)
print(summary.to_string())

def _series(roll, name):
    n = min(len(roll["dates"]), len(roll["values"]))
    return pd.Series(roll["values"][:n], index=pd.to_datetime(roll["dates"][:n]), name=name)

fig, ax = plt.subplots(figsize=(10, 4))
for roll, name in ((agent_roll, "EIIE"), (eq_roll, "equal-weight"), (rp_roll, "risk-parity")):
    _series(roll, name).plot(ax=ax)
ax.set_title("Test-window portfolio value")
ax.set_ylabel("value")
plt.tight_layout()
plt.show()
